# Elastic Net Regression: From First Principles

## 1. Why Combine Two Penalties?

Suppose we want to predict a numerical outcome from many measurements. Some measurements may be noisy; others may contain almost the same information. Ordinary Linear Regression chooses coefficients to reduce training error, but it does not directly discourage large coefficients. When predictors overlap, small changes in the sample can lead to very different allocations of coefficient weight.

For a concrete example, imagine predicting house prices from two related measures of living space, age, neighborhood indicators, and many additional attributes. We may want to retain information shared by the area measurements while removing features that add little predictive value.

Ridge Regression discourages large squared coefficients and often stabilizes correlated effects, but usually retains every feature. Lasso penalizes absolute coefficient values and can set coefficients exactly to zero. However, its choice among similar features can be unstable. **Elastic Net combines the two penalties**, allowing both coefficient sharing and sparse predictions.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Core intuition</strong><br>Use an L1 penalty to allow exact zeros and an L2 penalty to stabilize how coefficient weight is distributed. The balance is a modeling choice to validate, not a guarantee of better predictions.</div>

We will build the objective, explain how the two penalties work together, fit small examples, and then connect the theory to the companion's preprocessing and evaluation workflow.

## 2. Inputs, Outputs, and Essential Vocabulary

Elastic Net is a **supervised regression** method. It learns from features paired with known numerical targets. For a new row of features, it produces a numerical prediction. It does not directly output a class label or a class probability.

For $n$ training rows and $p$ features, its prediction for row $i$ is

$$
\hat y_i=b+\sum_{j=1}^{p}w_jx_{ij}.
$$

| Symbol or term | Meaning |
|---|---|
| $x_{ij}$ | Feature $j$ for training row $i$ |
| $y_i$, $\hat y_i$ | Observed target and predicted target |
| $w_j$ | Learned coefficient for feature $j$ |
| $b$ | Learned intercept or baseline |
| Residual $e_i=y_i-\hat y_i$ | The amount by which observation exceeds prediction |
| Regularization | A penalty that changes which fitted coefficients are preferred |
| Sparse model | A prediction equation containing many zero coefficients |
| Correlated predictors | Features whose values overlap systematically across rows |

Coefficients and the intercept are learned **parameters**. Penalty strength and mixture are **hyperparameters**, selected using validation. A zero coefficient removes a column's contribution from this fitted equation; it does not establish that the underlying variable is universally irrelevant.

The model stays linear in the supplied features. Powers and interactions can be added as features, but Elastic Net does not automatically discover arbitrary curves or interactions.

## 3. What L1 and L2 Each Contribute

The L1 penalty uses the sum of absolute coefficient values:

$$
\|\mathbf w\|_1=\sum_{j=1}^{p}|w_j|.
$$

Here $\mathbf w$ is the vector of all feature coefficients. Absolute values prevent positive and negative coefficients from canceling. The corner of absolute value at zero allows the optimum to set a coefficient exactly to zero.

The squared L2 penalty is

$$
\|\mathbf w\|_2^2=\sum_{j=1}^{p}w_j^2.
$$

Squaring makes large individual coefficients especially costly. For example, the allocations $(2,0)$ and $(1,1)$ both have L1 norm 2, but squared L2 norms 4 and 2. If their predictions are identical, the squared penalty favors sharing the weight.

| Model | Penalty | Typical effect |
|---|---|---|
| Ridge | Squared L2 | Shrinkage and more stable allocations; usually no exact zeros |
| Lasso | L1 | Shrinkage with the possibility of exact zeros |
| Elastic Net | L1 and squared L2 | Sparsity together with a preference for less concentrated weights |

Neither penalty directly counts the number of selected features. Nor does a large raw coefficient alone establish overfitting: coefficient magnitude also depends on measurement units. We will address scaling before practical fitting.

## 4. The Elastic Net Objective

We use the convention implemented by the companion's scikit-learn estimator:

$$
J(b,\mathbf w)=\frac{1}{2n}\sum_{i=1}^{n}(y_i-\hat y_i)^2
+\alpha r\sum_{j=1}^{p}|w_j|
+\frac{\alpha(1-r)}{2}\sum_{j=1}^{p}w_j^2.
$$

The first term is half the mean squared error. The positive constant $\alpha$ sets overall regularization strength; $r$, between 0 and 1, is the mixture parameter called `l1_ratio`. The intercept is not penalized.

It helps to define two effective strengths:

$$
\lambda_1=\alpha r,\qquad \lambda_2=\alpha(1-r).
$$

The objective is then half-MSE plus $\lambda_1\|\mathbf w\|_1+(\lambda_2/2)\|\mathbf w\|_2^2$. The division by 2 on the squared term makes its derivative simpler. This convention is specified in the [ElasticNet API documentation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html).

The model minimizes the sum of all three terms. It can accept extra training error to reduce penalties. At prediction time, only the linear prediction equation is evaluated; neither penalty is added to a predicted price.

For $\alpha>0$, we can also recover $\alpha=\lambda_1+\lambda_2$ and $r=\lambda_1/(\lambda_1+\lambda_2)$. These identities apply to the stated convention, including the factor of one half on the squared term.

## 5. Understanding Alpha, Mixture, and Endpoints

| Setting | Mathematical result | Interpretation |
|---|---|---|
| $\alpha=0$ | Ordinary least-squares objective | No regularization; the mixture has no effect |
| $r=1$, $\alpha>0$ | Lasso objective | Only L1 regularization |
| $r=0$, $\alpha>0$ | Ridge-type objective | Only squared L2 regularization |
| $0<r<1$, $\alpha>0$ | Both penalties | Shrinkage, possible zeros, and stabilized allocations |

At fixed $r$, increasing alpha strengthens both penalties. Too much regularization can suppress useful relationships and underfit. At fixed alpha, increasing $r$ strengthens L1 while weakening L2. The resulting feature count and individual coefficients need not change monotonically on correlated data.

At `l1_ratio=0.5`, the effective strengths $\lambda_1$ and $\lambda_2$ are equal. This does not mean half the features are selected or that the two penalty terms contribute equally to the objective: their coefficient functions differ.

The Ridge endpoint requires care in scikit-learn. At $r=0$, multiply the entire objective by $2n$:

$$
2nJ=\mathrm{SSE}+n\alpha\sum_jw_j^2.
$$

Scikit-learn's `Ridge` uses SSE plus its alpha times the squared norm. Thus, with identical unweighted data and preprocessing, the matching Ridge strength is $\alpha_{\mathrm{Ridge}}=n\alpha_{\mathrm{ElasticNet}}$, not the same numeric alpha. At $r=1$, the Lasso convention matches directly. Use dedicated `Ridge` or `LinearRegression` estimators for the pure L2 or unregularized baselines.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Common pitfall</strong><br>The mixture blends penalties inside one optimization problem. Elastic Net is not an average of separately fitted Ridge and Lasso predictions.</div>

## 6. Why Correlated Features Can Share Weight

Retain the original notes' duplicated-feature example: suppose $x_1=x_2=x$ and the target follows $y=2x$. The allocations $(2,0)$ and $(1,1)$ both produce $2x$ before regularization. Training error cannot distinguish them.

Among nonnegative pairs with a fixed sum of 2, the L1 penalty is always 2. The squared L2 penalty is smallest for $(1,1)$. Therefore, with a positive L2 component, Elastic Net favors the balanced allocation among these equally predictive candidates. The actual fitted sum can also shrink; comparing allocations at a fixed sum does not finish the fit.

For exact duplicate columns treated identically, positive L2 regularization makes the optimal weights equal. With merely similar columns, their weights may be similar but need not be identical. Differences in scaling, noise, or relationships with other features matter. Elastic Net does not guarantee that an entire correlated group will be retained.

The objective is convex, so there are no inferior local minima. When $\lambda_2>0$, the objective has positive curvature in every coefficient direction, yielding a unique coefficient solution even with redundant columns. Pure Lasso can have multiple coefficient solutions. Uniqueness is a mathematical property of the fit, not proof that its coefficients reveal causal effects.

## 7. How One Coefficient Is Updated

To turn these intuitions into a fitting method, center each feature and the target using training means. Let $z_{ij}$ denote centered feature $j$ and $v_i=y_i-\bar y$ the centered target. The bar denotes the training average. Scaling may also be applied; the derivation allows general feature variation.

While updating coefficient $w_j$, hold the others fixed and calculate its **partial residual**:

$$
r_i^{(j)}=v_i-\sum_{k\ne j}w_kz_{ik}.
$$

This removes the predictions from all other features, leaving what feature $j$ can try to explain. Define

$$
a_j=\frac1n\sum_i z_{ij}^2,\qquad
c_j=\frac1n\sum_i z_{ij}r_i^{(j)}.
$$

Here $a_j$ measures feature variation and $c_j$ measures its alignment with the remaining target signal. The terms involving $w_j$ in the objective are

$$
\frac{a_j+\lambda_2}{2}w_j^2-c_jw_j+\lambda_1|w_j|.
$$

For positive $w_j$, setting the derivative to zero gives $(c_j-\lambda_1)/(a_j+\lambda_2)$. For negative $w_j$, it gives $(c_j+\lambda_1)/(a_j+\lambda_2)$. At zero, the corner in absolute value makes zero optimal when $|c_j|\leq\lambda_1$.

Combine the cases with the soft-threshold function

$$
S(c,t)=\begin{cases}
c-t,&c>t,\\
0,&|c|\leq t,\\
c+t,&c<-t,
\end{cases}
\qquad
w_j\leftarrow\frac{S(c_j,\lambda_1)}{a_j+\lambda_2}.
$$

The arrow means replace the current coefficient. L1 determines the threshold in the numerator; L2 adds shrinkage through the denominator. For a standardized nonconstant feature, $a_j=1$. Constant features carry no variation and should be removed or handled explicitly.

## 8. Coordinate Descent: From Updates to a Fitted Model

**Coordinate descent** repeats the preceding update one feature at a time:

1. Initialize the coefficients, for example at zero.
2. Compute a feature's partial residual using the current other coefficients.
3. Soft-threshold its residual alignment and divide by its variation plus the L2 strength.
4. Update the next feature, then make additional passes through all features.
5. Stop when the solver's convergence criterion is met, or report that the iteration limit was reached.

With correlated columns, a later update changes the residual relevant to an earlier feature. Multiple passes are therefore needed. Independently thresholding each feature's original correlation with the target is generally not the joint solution.

After working with centered features, restore the intercept:

$$
b=\bar y-\sum_jw_j\bar x_j.
$$

The feature means here belong to the representation used for fitting. Standardized training features have mean zero, so their fitted intercept is the training target mean when the target is not transformed.

The [scikit-learn implementation](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ElasticNet.html) uses coordinate descent. `max_iter` limits iterations, while `tol` controls stopping accuracy. A large iteration limit is not itself evidence of convergence. Inspect warnings and diagnostics such as `n_iter_` and `dual_gap_`, an optimization-gap diagnostic rather than a prediction-error metric.

## 9. Worked Example: Thresholding and Shrinkage Together

Use four hypothetical houses with already standardized features $z_1$ and $z_2$. Think of these as an area measurement and another numerical attribute. Price is in tens of thousands of dollars. This deliberately simple data has orthogonal columns: their cross-product sum is zero.

| House | $z_1$ | $z_2$ | Price $y$ | Centered price $v=y-30$ |
|---|---:|---:|---:|---:|
| A | -1 | -1 | 27 | -3 |
| B | -1 | 1 | 29 | -1 |
| C | 1 | -1 | 31 | 1 |
| D | 1 | 1 | 33 | 3 |

Both feature means are zero and $a_1=a_2=1$. Orthogonality means changing one coefficient does not change the alignment for the other. We can therefore solve independently:

$$
c_1=\frac{3+1+1+3}{4}=2,\qquad
c_2=\frac{3-1-1+3}{4}=1.
$$

Choose $\alpha=2$ and $r=0.5$, giving $\lambda_1=1$ and $\lambda_2=1$. The update yields

$$
w_1=\frac{S(2,1)}{1+1}=\frac12,\qquad
w_2=\frac{S(1,1)}{1+1}=0.
$$

The intercept is 30, and the model is $\hat y=30+0.5z_1$. L1 removes the second coefficient; L2 further shrinks the surviving coefficient. Pure Lasso with L1 strength 1 would give $(1,0)$; pure L2 with $\lambda_2=1$ in our convention would give $(1,0.5)$. These comparisons hold one effective penalty strength fixed, not the overall alpha across all models.

## 10. Check the Objective and Make a Prediction

For the fitted Elastic Net model:

| House | Actual price | Prediction | Residual | Squared residual |
|---|---:|---:|---:|---:|
| A | 27 | 29.5 | -2.5 | 6.25 |
| B | 29 | 29.5 | -0.5 | 0.25 |
| C | 31 | 30.5 | 0.5 | 0.25 |
| D | 33 | 30.5 | 2.5 | 6.25 |

The SSE is 13. Half-MSE is $13/(2\cdot4)=1.625$. The L1 penalty is $1(0.5)=0.5$, and the squared L2 penalty is $(1/2)(0.5^2)=0.125$. Therefore

$$
J=1.625+0.5+0.125=2.25.
$$

Ordinary least squares uses coefficients $(2,1)$ and fits these rows perfectly. Evaluating those coefficients under the same Elastic Net objective gives $0+3+5/2=5.5$. An intercept-only prediction of 30 gives SSE 20 and objective $20/8=2.5$. The fitted Elastic Net solution has the smaller combined objective, although it has more training error than OLS.

For a new house with standardized features $(0.5,-0.5)$, the prediction is $30+0.5(0.5)=30.25$, or $302{,}500$. Its actual price is unknown, so we cannot infer prediction quality.

<div style="color:#17324d; background-color:#eaf4ff; border-left:5px solid #2878b5; padding:10px 14px; margin:12px 0;"><strong>Worked-example takeaway</strong><br>Thresholding and additional shrinkage are separate effects. A lower penalized training objective does not establish better future predictions, and a removed feature can still contribute to the underlying relationship.</div>

## 11. A Second Small Example: Actual Sharing Between Duplicates

Return to the original duplicated-feature example, now complete the optimization. Let both centered feature columns equal $z$, with mean square one, and let the centered target be $v=2z$. Choose $\lambda_1=\lambda_2=1$ as above.

The positive squared penalty gives a unique solution, and exchanging the identical columns cannot change the problem. Their fitted coefficients must therefore be equal: write $w_1=w_2=q$. The objective reduces to

$$
J(q)=\tfrac12(2-2q)^2+2|q|+q^2.
$$

For $q>0$, the derivative is $-4+4q+2+2q=6q-2$. Setting it to zero gives $q=1/3$. This is positive and satisfies the assumed branch; convexity makes it the global minimum. Thus the fitted total effect is $2/3$, shared equally, rather than the unpenalized total of 2.

At $q=1/3$, the objective is $8/9+2/3+1/9=5/3$. An allocation $(2/3,0)$ makes the same predictions and has the same L1 penalty, but a larger squared L2 penalty, producing objective $16/9$.

This example separates two ideas: the fit shrinks the total effect, and the L2 component prefers sharing that effect across duplicate features. Neither behavior proves that duplicating a feature is useful preprocessing; adding duplicate columns changes the regularized problem.

## 12. Scaling and Representation Are Part of the Model

If a feature is multiplied by 1000, the coefficient needed for unchanged predictions is divided by 1000. Its L1 penalty becomes 1000 times smaller and its squared penalty becomes one million times smaller. A shared penalty has a different meaning when measurement units change.

A common preparation is standardization:

$$
z_{ij}=\frac{x_{ij}-\mu_j}{s_j},
$$

where $\mu_j$ and $s_j$ are feature $j$'s training mean and standard deviation. A unit change then means one training standard deviation. This makes coefficient penalties easier to compare, but does not decide which features are useful.

Handle missing values with a suitable training-fitted strategy, inspect outliers, and remove or explicitly handle constant features. Squared-error fitting and ordinary standardization can both be influenced by extreme observations; regularization is not robust outlier handling.

Encode categorical features meaningfully, for example with one-hot indicators. Arbitrary numeric category labels impose an unjustified ordered effect. Scaling indicators, choosing a reference category, and adding interactions can all change the penalized fit. Sparsity applies to represented columns, so selecting one indicator does not necessarily select an entire categorical variable.

<div style="color:#22543d; background-color:#edf8f1; border-left:5px solid #2f855a; padding:10px 14px; margin:12px 0;"><strong>Practical tip</strong><br>Fit imputation, encoding, scaling, and Elastic Net within each training fold. Apply those fitted transformations to that fold's validation rows and later to new data.</div>

Changing target units also changes the balance of the penalties at fixed settings. Retune both hyperparameters after changing the target representation rather than assuming the old values carry over.

## 13. Tune Both Parameters Without Leaking Information

Training error cannot choose a useful penalty: removing regularization permits the smallest training squared error. Instead, evaluate predictions on rows not used in each fit.

1. Define the target, prediction-time features, and primary metric. Exclude information generated after the target outcome.
2. Reserve a final test set. Respect time order for forecasting and keep related subjects or groups together when necessary.
3. Build a complete preprocessing-and-model pipeline. Within development training data, cross-validate combinations of alpha and mixture, fitting all preprocessing separately inside each fold.
4. Inspect average validation error and variation across folds. Expand or refine the candidate ranges if the results justify it.
5. Select the configuration using validation, refit the complete pipeline on all development training data, and evaluate once on the reserved test set.
6. Apply the saved transformations and model to future rows with the same feature definitions and ordering.

For the companion's pipeline names, the search parameters are `model__alpha` and `model__l1_ratio`. One illustrative starting grid uses alpha values $0.01,0.1,1,10$ with ratios $0.1,0.5,0.9,1$. These are search candidates, not universally good settings. Compare pure Ridge separately. Very small positive mixture values may require a carefully specified alpha range and numerical checks.

Because the parameters interact, selecting the mixture at one arbitrary alpha can miss a better combination elsewhere. Do not select features once using all targets before cross-validation either: feature selection is learned from data and belongs inside the validation process.

If sparse deployment is valuable, assess the tradeoff explicitly alongside prediction error. A smaller selected set is not automatically the best model, and selecting different correlated representatives across folds can reveal unstable interpretation even when prediction scores are stable.

## 14. Read the Companion as an Illustration

`05_Elastic_Net_Regression.ipynb` loads `load_diabetes(as_frame=True)` and makes an 80/20 training/test split with `random_state=42`. It fits pipelines containing `StandardScaler` and one of three estimators:

| Estimator | Settings shown |
|---|---|
| Ridge | `alpha=1.0` |
| Lasso | `alpha=1.0`, `max_iter=10000` |
| Elastic Net | `alpha=1.0`, `l1_ratio=0.5`, `max_iter=10000` |

These pipelines correctly learn their scaling from the rows passed to `fit`. Their equal numeric alphas do not imply equivalent penalty strengths or a fair comparison of optimally tuned models.

The companion reports test MAE, RMSE, and $R^2$, then displays Elastic Net coefficients with `selected = coef != 0`. That flag identifies nonzero returned coefficients, including tiny nonzero values. It does not measure practical importance. A zero also does not necessarily remove the original input column from the pipeline's expected schema.

Finally, it compares ratios $0.1,0.3,0.5,0.7,0.9$ at fixed alpha 0.5 on the same test set. This shows how the mixture changes a fit, but does not jointly tune the two parameters.

<div style="color:#664b12; background-color:#fff5df; border-left:5px solid #b7791f; padding:10px 14px; margin:12px 0;"><strong>Keep the test set independent</strong><br>Selecting the best ratio from the companion's test-score table would turn that test set into tuning data. Use training-only cross-validation for selection and reserve test results for the final assessment.</div>

These notes describe the source code. They do not claim measured scores, selected features, convergence, or a winning model because the companion was not executed for this revision.

## 15. Evaluate Predictions and Interpret Coefficients

For $m$ evaluation rows with residuals $e_i=y_i-\hat y_i$:

$$
\mathrm{MAE}=\frac1m\sum_i|e_i|,\qquad
\mathrm{MSE}=\frac1m\sum_i e_i^2,\qquad
\mathrm{RMSE}=\sqrt{\mathrm{MSE}}.
$$

MAE describes average absolute error in target units. RMSE also has target units but gives large errors more influence. Lower is better on the same evaluation rows and target representation. Neither metric includes a coefficient penalty.

For nonconstant evaluation targets,

$$
R^2=1-\frac{\sum_i(y_i-\hat y_i)^2}{\sum_i(y_i-\bar y_{\mathrm{eval}})^2},
$$

where $\bar y_{\mathrm{eval}}$ is their mean. One is perfect, zero matches the evaluation-mean reference in this formula, and negative values are worse. Constant targets make the mathematical ratio undefined. Also compare with a deployable baseline that learns the training target mean.

Inspect residuals against predictions and features. Curvature can indicate missing nonlinear terms; error patterns across groups or time can expose weaknesses hidden by averages. A small advantage on one split is limited evidence of a reliable improvement.

For standardized features, $w_j$ is the change in predicted target per one-training-standard-deviation increase, holding other represented features fixed. With an untransformed target, raw-unit coefficients and intercept are

$$
\beta_j=\frac{w_j}{s_j},\qquad
b_{\mathrm{raw}}=b_z-\sum_j\frac{w_j\mu_j}{s_j},
$$

where $b_z$ is the intercept in standardized coordinates. These transformations preserve predictions; they do not remove shrinkage bias. Coefficient magnitude is not a complete importance ranking, and holding correlated features fixed may describe unrealistic cases. Predictive associations are not causal effects or automatic statistical significance findings.

## 16. Tradeoffs, Alternatives, and Computational Cost

Shrinkage can reduce variation in predictions across training samples while introducing systematic bias toward smaller effects. Removing weak contributions can improve prediction when it reduces variance enough, but can hurt when many small effects truly matter. Noise that the available features cannot explain remains.

| Candidate | When it is worth comparing |
|---|---|
| Ordinary least squares | A stable linear fit may need little regularization |
| Ridge | Many modest effects may matter and sparsity is less important |
| Lasso | A sparse representation is plausible and selection is sufficiently stable |
| Elastic Net | Both sparsity and correlated predictors matter |
| Transformed features plus a linear model | Known curves or interactions can be represented explicitly |
| Trees or forests | Important nonlinear relationships are difficult to specify as linear features |

Elastic Net adds a tuning dimension compared with Ridge or Lasso. More candidate combinations mean more fits and more opportunity to overfit validation results, especially with small datasets. Use a reasonable search and keep the final test assessment separate.

For dense data with maintained residuals, a coordinate-descent pass can require roughly $O(np)$ work. Total cost depends on the number of passes, candidates, folds, and numerical implementation. Correlation and weak regularization can slow convergence. A dense prediction takes roughly $O(p)$ arithmetic per row; a sparse coefficient vector helps only if the deployment implementation uses that sparsity.

`max_iter` and `tol` control numerical computation, not the statistical tradeoff. Changing them to silence a warning without understanding convergence is different from selecting alpha and the mixture for predictive performance.

## 17. Assumptions, Failure Modes, and Recap

Elastic Net does not require normally distributed errors merely to compute a fit. Useful predictions require informative features, a suitable representation of the relationship, and training data relevant to future cases. Ordinary random-split evaluation also relies on rows being sufficiently independent; time dependence and repeated subjects need appropriate splitting.

| Failure mode | Why the combined penalty is insufficient | Practical response |
|---|---|---|
| Excessive shrinkage | Both useful and weak effects can be suppressed | Revisit joint tuning and validation |
| Missing nonlinear structure | Penalties do not create new features | Engineer justified transformations or compare other models |
| Outliers or bad measurements | Squared error remains sensitive | Inspect data and consider robust alternatives |
| Unstable selected groups | Correlation and sampling uncertainty remain | Examine resampling stability and interpretation limits |
| Leakage or distribution shift | The evaluation may not reflect future use | Audit features, splits, and later performance |
| Need for guaranteed variable recovery | Sparse coefficients are not proof of true relevance | Avoid treating selection as causal discovery |

A positive L2 component gives a unique coefficient solution, but uniqueness does not guarantee good generalization. Elastic Net can extrapolate through its linear equation, but such extrapolation still needs justification. Adding both penalties is not automatically better than either alone.

Elastic Net fits a linear prediction equation with two linked controls: L1 provides a threshold for exact zeros, and L2 shrinks and stabilizes coefficient allocations. Choose scaling and feature representation deliberately, tune strength and mixture together, and evaluate predictions independently of the training penalties.

The main caution is interpretation: shared weights and a sparse feature list describe one regularized fit. They do not establish which variables truly cause the outcome or guarantee that the same selection will persist in new data.